Description: From peak information, plot the relative start time of the peaks and get the afterpulsing rate

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import glob
import os
sys.path.insert(0,"/home/ws/sk6801/sw/UCSD_analysis/sandpro")
import sandpro
import configparser
import json
import scipy.stats
from matplotlib.colors import LogNorm

from scipy.optimize import curve_fit
import datetime
import pandas as pd
from copy import deepcopy
from numba import jit
import time

sys.path.insert(0,"../src/")
import common.d2d as d2d
import common.utils as util
import data_processing.fast_processor_all_channel as fast_processor
from data_processing.event_processor_all_channel import EventProcessor
from application.peak_processor import Peak_Processor

from data_structure.waveform_info import WaveformInfo
from data_structure.peak_info import PeakInfo

%run "plot_style_kalinka.py"


### Import and Process Data

In [ ]:
df_SPE_position = pd.read_csv(
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/spe_position_LXe_2.csv",
                 delimiter=",")

df_SPE_position

In [ ]:
peak_processor = Peak_Processor()

df = pd.read_csv(
    "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/kalinka_20250616_LXe_gain_info_single_channel.csv", 
                 parse_dates=["date_time"],
                 delimiter=",",
                 quotechar='"', 
                 skipinitialspace=True, 
                 encoding="utf-8")


# convert string to array
if isinstance(df['board_0_channels'][0], str):
    if "," in df['board_0_channels'][0]:
        df['board_0_channels'] = df['board_0_channels'].apply(json.loads).apply(np.array)
    else:
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace("  ", ","))
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace("[ ", "["))
        df['board_0_channels'] = df['board_0_channels'].apply(lambda x: x.replace(" ", ","))
        df['board_0_channels'] = df['board_0_channels'].apply(json.loads).apply(np.array)

        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace("  ", ","))
        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace("[ ", "["))
        df['board_1_channels'] = df['board_1_channels'].apply(lambda x: x.replace(" ", ","))
        df['board_1_channels'] = df['board_1_channels'].apply(json.loads).apply(np.array)

print(len(df))
idx_nan = np.where((df['spe_position'].isna()) & (df['voltage_preamp1_V']<-46))[0]
# replace NaN values with SPE position
voltage_array = df.loc[idx_nan, 'voltage_preamp1_V']
channel_array = df.loc[idx_nan, 'channel']
print(len(idx_nan))
for idx, voltage, channel in zip(idx_nan, voltage_array, channel_array):
    # find the SPE position for the given voltage and channel
    df.loc[idx, 'spe_position'] = df_SPE_position[
        (df_SPE_position['voltage_preamp1_V'] == voltage) & 
        (df_SPE_position['channel'] == channel)
    ]['spe_position'].values
    df.loc[idx, 'spe_position_err'] = df_SPE_position[
        (df_SPE_position['voltage_preamp1_V'] == voltage) & 
        (df_SPE_position['channel'] == channel)
    ]['spe_position_err'].values


In [ ]:
#### Basic Data Selection

all_runs_d2d = d2d.data(df)


mask_data_taking_mode = (all_runs_d2d.data_taking_mode == "all_channels")
mask_na = ~np.isnan(all_runs_d2d.spe_position)
# mask = ~np.isnan(all_runs_d2d.gain)
mask_voltage = (all_runs_d2d.voltage_preamp1_V < -46) & (all_runs_d2d.voltage_preamp1_V > -52)
mask_post_trigger = (all_runs_d2d.post_trigger == 70)
mask_run_tag = (all_runs_d2d.run_tag == "LXe/tritium")
# mask_run_tag = (all_runs_d2d.run_tag == "LXe/gain_calibration")
mask_comment = (~util.vec_regex_search('trash', all_runs_d2d.comment)) & (~util.vec_regex_search('test', all_runs_d2d.comment))

mask = mask_data_taking_mode & mask_na & mask_run_tag & mask_voltage & mask_post_trigger & mask_comment
# & mask_voltage

all_runs_d2d.apply_mask(mask, inplace=True, dry = False)
all_run_list = np.unique(all_runs_d2d.md_full_path)



#### Run Selection

check how many runs have all 24 channel data, print a list

In [ ]:
count_successful = 0
count_failed = 0

count_n_events = 0

for i, md_full_path in enumerate(all_run_list):

    mask = all_runs_d2d.md_full_path == md_full_path
    single_run = all_runs_d2d.apply_mask(mask, inplace=False, dry = True)

    if len(single_run.channel) < 24:
        # print(f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels. Run tag: {single_run.run_tag[0]}. Comment: {single_run.comment[0]}. Voltage: {single_run.voltage_preamp1_V[0]}")
        count_failed += 1

        continue    
    elif len(single_run.channel) == 24:
        print(f"Run {i}: {md_full_path} has 24 channels. Run tag: {single_run.run_tag[0]}. Voltage: {single_run.voltage_preamp1_V[0]}. Comment: {single_run.comment[0]}. Event number: {single_run.n_processed_events[0]}")
        count_successful += 1
        count_n_events += single_run.n_processed_events[0]
        
    else: 
        raise ValueError(f"Run {md_full_path} has {len(single_run.channel)} channels, expected 24 channels.")

print(f"Total runs: {len(all_run_list)}"
      f"Successful runs: {count_successful} "
      f"Failed runs: {count_failed} "
      f"Success rate: {count_successful/len(all_run_list)*100:.2f}%")

print(f"Total number of events: {count_n_events}")

#### Save data to csv (enable if need to regenerate the file)

select runs from the list above -> histogram relative start time -> fit exponential function to the histogram -> save tau to output.csv

for more details can refer to peak_all_channel.ipynb, section "area vs time"

In [ ]:
# # ### Choose a run from the list above
# # initialize the result storage
# df_result = pd.DataFrame(columns=WaveformInfo().__dict__.keys())
# output_fname = "/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/afterpulse_test.csv"


# # for run_id in np.arange(len(all_run_list)):
# for run_id in [86,87,88,89,90]:
#     single_info_list = []
#     for md_full_path in all_run_list[run_id:run_id+1]:
#     # for md_full_path in all_run_list:
#         result, waveform, baseline, baseline_std = peak_processor.get_peak_level_data(
#             all_runs_d2d=all_runs_d2d,
#             md_full_path=md_full_path,
#             peak_merge_window_sample=0
#         )
#         single_info_list += result

#     df_result = pd.DataFrame.from_dict(single_info_list)

#     df_result['first_peak_start_time'] = df_result.groupby('event_id')['peak_rel_start_time_s'].transform('min')
#     df_result['time_diff'] = df_result['peak_rel_start_time_s'] - df_result['first_peak_start_time']

#     # peak_rel_start_time = d2d_data.peak_rel_start_time_s
#     time_diff = df_result['time_diff']

#     hist, bin_edges = np.histogram(time_diff*1e6, bins=100, range=[0,4])
#     bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
#     hist_log = np.log10(hist + 1)  # add 1 to avoid log(0)
#     # ax.plot(bin_centers, hist_log, 'b-', label='Data')

#     # fit the decay trend
#     mask = (bin_centers >= 1) & (bin_centers <= 2.5)
#     # linear fit
#     p, cov = np.polyfit(bin_centers[mask], hist_log[mask], deg=1, cov=True)


#     # find tau
#     tau = -1 / p[0]
#     a = np.exp(p[1])
#     # error of tau
#     tau_err = np.sqrt(cov[0, 0]) / p[0]**2


#     assert len(np.unique(df_result.md_full_path)) == 1


#     df_output = pd.DataFrame({
#         'file_name': df_result.md_full_path[0],
#         'run_tag': df_result.run_tag[0],
#         'comment': df_result.comment[0],
#         'datetime': df_result.date_time[0],
#         'voltage': np.unique(df_result.voltage_preamp1_V)[0],
#         'tau': tau,
#         'error': tau_err
#     },
#     index=[0]
#     )


#     with open(output_fname, 'a') as f:
#         df_output.to_csv(f, mode='a', header=f.tell() == 0, index=False)



#### Read from the generate afterpulse datafile

In [ ]:
afterpulse_df = pd.read_csv(
    '/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/processed_data/afterpulse_3.csv',
    parse_dates=["datetime"])
afterpulse_df['date'] = afterpulse_df['datetime'].dt.date


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))


for voltage in np.unique(afterpulse_df['voltage']):
    mask = afterpulse_df['voltage'] == voltage
    y_var = afterpulse_df['tau'][mask]
    y_var_err = afterpulse_df['error'][mask]
    x_var = (afterpulse_df['datetime'][mask] - afterpulse_df['datetime'][mask].iloc[0]).dt.total_seconds()


    # correlation
    corr = y_var.corr(x_var, method = 'pearson')
    # standard error of correlation
    covariance = y_var.cov(x_var)
    stdx = y_var.std()
    stdy = x_var.std()
    r = covariance / (stdx * stdy)
    corr_err = np.sqrt((1 - r**2) / (len(y_var) - 2))
    print(f"Correlation between Tau and Time: {corr:.2f} ± {corr_err:.2f}")

    plt.errorbar(
        afterpulse_df['datetime'][mask],
        y_var, 
        yerr=y_var_err, 
        label=f'{voltage} V: {corr:.2f} ± {corr_err:.2f}',
        fmt='o',
        )
    
plt.legend(
    title="Bias voltage: correlation",
    bbox_to_anchor=(0., 1),
    loc='upper left',
    frameon=True
)
plt.xlabel("Date")
plt.ylabel("$\\tau$ [$\mu$s]")

# rotate x-axis labels
plt.xticks(rotation=45)


# fname = f"/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/afterpulse_vs_time.pdf"
# plt.savefig(fname, dpi=300, bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 7))

for date in np.unique(afterpulse_df['date']):
    mask = afterpulse_df['date'] == date
    y_var = afterpulse_df['tau'][mask]
    y_var_err = afterpulse_df['error'][mask]
    x_var = afterpulse_df['voltage'][mask]

    # correlation
    corr = y_var.corr(x_var, method = 'pearson')
    # standard error of correlation
    covariance = y_var.cov(x_var)
    stdx = y_var.std()
    stdy = x_var.std()
    r = covariance / (stdx * stdy)
    corr_err = np.sqrt((1 - r**2) / (len(y_var) - 2))
    print(f"Correlation between Tau and Time: {corr:.2f} ± {corr_err:.2f}")

    plt.errorbar(
        x_var,
        y_var, 
        yerr=y_var_err, 
        label=f'{date}: {corr:.2f} ± {corr_err:.2f}',
        fmt='o',
        )
    
plt.legend(
    title="Date: correlation",
    bbox_to_anchor=(0., 0.),
    loc='lower left',
    frameon=True
)
plt.xlabel("Bias voltage")
plt.ylabel("$\\tau$ [$\mu$s]")

plt.ylim(2., 5.2)


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15, 6), sharey=True)
plt.style.use('tableau-colorblind10')
colors = plt.rcParams["axes.prop_cycle"]()
fig.subplots_adjust(wspace=0)

for voltage in np.unique(afterpulse_df['voltage'][afterpulse_df['voltage']>-52]):
    mask = afterpulse_df['voltage'] == voltage
    y_var = afterpulse_df['tau'][mask]
    y_var_err = afterpulse_df['error'][mask]
    x_var = (afterpulse_df['datetime'][mask] - afterpulse_df['datetime'][mask].iloc[0]).dt.total_seconds()


    # correlation
    corr = y_var.corr(x_var, method = 'pearson')
    # standard error of correlation
    covariance = y_var.cov(x_var)
    stdx = y_var.std()
    stdy = x_var.std()
    r = covariance / (stdx * stdy)
    corr_err = np.sqrt((1 - r**2) / (len(y_var) - 2))


    color = next(colors)["color"]


    ax[0].errorbar(
        afterpulse_df['datetime'][mask],
        y_var, 
        yerr=y_var_err, 
        label=f'{voltage} V: {corr:.2f} ± {corr_err:.2f}',
        fmt='o',
        color = color
        )

    mask = mask & (afterpulse_df['datetime'] > pd.Timestamp('2024-10-29'))

    ax[0].errorbar(
        afterpulse_df['datetime'][mask],
        y_var[mask], 
        yerr=y_var_err[mask], 
        fmt='o',
        color = color,
        markerfacecolor = 'white'
        )
    


#rotate axis ticks lable
xlabels = ax[0].get_xticklabels()
ax[0].set_xticklabels(xlabels, rotation=30, ha='right')

for date in np.unique(afterpulse_df['date']):
    mask = afterpulse_df['date'] == date
    y_var = afterpulse_df['tau'][mask]
    y_var_err = afterpulse_df['error'][mask]
    x_var = abs(afterpulse_df['voltage'][mask])

    # correlation
    corr = y_var.corr(x_var, method = 'pearson')
    # standard error of correlation
    covariance = y_var.cov(x_var)
    stdx = y_var.std()
    stdy = x_var.std()
    r = covariance / (stdx * stdy)
    corr_err = np.sqrt((1 - r**2) / (len(y_var) - 2))

    

    color = next(colors)["color"]

    if (afterpulse_df['run_tag'][mask].iloc[0] == "LXe/tritium"):
        # marker hollow
        markerfacecolor='white'
    else:
        markerfacecolor=color

    ax[1].errorbar(
        x_var,
        y_var, 
        yerr=y_var_err, 
        label=f'{date}: {corr:.2f} ± {corr_err:.2f}',
        fmt='o',
        color=color,
        mfc=markerfacecolor
    )

ax[0].legend(
    title="bias voltage: correlation",
    bbox_to_anchor=(0.5, 1.),
    loc='upper center',
    frameon=True,
    ncols=2,
    fontsize=14

)


ax[1].legend(
    title="date: correlation",
    bbox_to_anchor=(0.5, 1),
    loc='upper center',
    frameon=True,
    ncols=2,
    fontsize=14

)

plt.ylim(2.5, 8.2)

ax[0].set_ylabel("$\\tau$ [$\mu$s]")
ax[0].set_xlabel("Date")
ax[1].set_xlabel("$|V_{bias}|$ [V]")

# fname = f"/kalinka/storage/darkmatter/XENONnT/sk6801/UCSD_data/plots/afterpulse_dependence.pdf"
# plt.savefig(fname, dpi=300, bbox_inches='tight')